# 05 Submit: three model families on the hidden test set

The final submission. It reads the fold models of three families trained in notebook 04 and its sister runs: RadImageNet ResNet50 with spatial pooling (`radcs`), RadImageNet ResNet50 with average pooling (`radcr`) and ImageNet SE-ResNeXt50 (`sx50r`), five folds each, all trained on the raw fused labels. Nothing is trained here.

In a normal run Kaggle mounts three example test studies, so this run only checks that every step works. When the notebook is submitted, Kaggle runs it again on the hidden test set of about 1,300 studies, which builds the test volumes, decides each knee's side, predicts with all 15 models and writes `submission.csv`. Submitted as version 1, this notebook scored 0.940 on the public leaderboard; this version adds only the markdown. The code is the same as in the training notebooks, and the settings in the first cell select inference only.

Imports, the data path, the model families and this run's settings: the three families with equal weight (`FAM_WEIGHT`), `TRAIN = False` so that a missing fold model is an error rather than a reason to train, and no models trained in the scoring run (`SUB_TRAIN = []`).

In [1]:
import gc, glob, hashlib, math, os, subprocess, sys, time
from functools import partial
from multiprocessing import Pool
import cv2, numpy as np, pandas as pd, pydicom, timm, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold

T0 = time.time()
R = next((p for p in ("/kaggle/input/competitions/rsna-knee-abnormality-detection", "/kaggle/input/rsna-knee-abnormality-detection")
          if os.path.exists(f"{p}/train.csv")), "/kaggle/input/competitions/rsna-knee-abnormality-detection")
print("data root", R, flush=True)
HIDDEN = os.path.exists(f"{R}/test.csv") and len(pd.read_csv(f"{R}/test.csv")) > 10
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",
       "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
FAMILIES = {"resnet50": ("resnet50", "v2", 224), "r50v3": ("resnet50", "v3", 224), "coat0": ("coatnet_0_rw_224", "v3", 224), "r50v4": ("resnet50", "v4", 320), "eb3v3": ("efficientnet_b3", "v3", 224), "sx50v3": ("seresnext50_32x4d", "v3", 224), "dinosv3": ("vit_small_patch14_dinov2", "v3", 224),
            "sx50r2": ("seresnext50_32x4d", "v3", 224), "r50r2": ("resnet50", "v3", 224),
            "radr50v3": ("resnet50@rad", "v3", 224), "radr2": ("resnet50@rad", "v3", 224),
            "rdn121": ("densenet121@rad", "v3", 224),
            "radc": ("resnet50@rad+c", "v3", 224), "rdn121c": ("densenet121@rad+c", "v3", 224),
            "sx50n": ("seresnext50_32x4d@+n", "v3", 224), "radcn": ("resnet50@rad+cn", "v3", 224),
            "radcs": ("resnet50@rad+cs", "v3", 224), "radc4": ("resnet50@rad+c", "v4", 320),
            "sx50s": ("seresnext50_32x4d@+s", "v3", 224), "radcsn": ("resnet50@rad+csn", "v3", 224),
            "radc2": ("resnet50@rad+c", "v2", 224), "radcs2": ("resnet50@rad+cs", "v2", 224), "radcn2": ("resnet50@rad+cn", "v2", 224),
            "radcr": ("resnet50@rad+c", "v3", 224), "sx50r": ("seresnext50_32x4d", "v3", 224)}
FAM_WEIGHT = {"radcs": 1.0, "radcr": 1.0, "sx50r": 1.0}
FAM = "radcs"
ARCH, PREP, RES = FAMILIES[FAM]
FOLDS, SEED = 5, 42
RUN_FOLDS = [0, 1, 2, 3, 4]
EPOCHS, BS = 20, 8
LR_BB, LR_HEAD = 2e-4, 1e-3
WIN = [5, 2, 4, 2, 3]
STRAT = False
EVAL_DRAWS = 0
TTA_REV = False
EVAL_CACHED = False
TEACHERS, DISTILL = [], 0.6
TRAIN = False
SUB_TRAIN, FIT_H = [], {"r50v3": 1.5, "sx50v3": 1.8, "dinosv3": 1.8, "sx50r2": 1.8, "r50r2": 1.5, "radr50v3": 1.5, "radr2": 1.5, "rdn121": 2.0, "radc": 1.5, "rdn121c": 2.0, "sx50n": 1.8, "radcn": 1.5, "radcs": 1.6, "radc4": 3.2}
RAM_GUARD = 12.0
LR_ARCH = {"vit_small_patch14_dinov2": 5e-5}
LIMIT = 0
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if DEV == "cuda":
    print(torch.cuda.get_device_name(0), torch.cuda.device_count())


def ram_gb():
    m = {l.split(":")[0]: int(l.split()[1]) for l in open("/proc/meminfo") if l.split(":")[0] in ("MemTotal", "MemAvailable")} if os.path.exists("/proc/meminfo") else {}
    return (m.get("MemTotal", 0) - m.get("MemAvailable", 0)) / 2**20, m.get("MemTotal", 0) / 2**20


def ram():
    used, total = ram_gb()
    return f"ram {used:.1f}/{total:.1f}G"


def find(name):
    hits = [p for top in ("notebooks", "datasets", "models") for p in glob.glob(f"/kaggle/input/{top}/**/{name}", recursive=True)]
    return sorted(set(hits + glob.glob(f"/kaggle/input/*/{name}") + glob.glob(f"/kaggle/input/*/*/{name}") + glob.glob(f"/kaggle/working/{name}")))


for w in find("wheels"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", w, "pylibjpeg",
                    "pylibjpeg-libjpeg", "pylibjpeg-openjpeg", "python-gdcm"], check=False)

data root /kaggle/input/competitions/rsna-knee-abnormality-detection
Tesla T4 2


ERROR: Could not find a version that satisfies the requirement pylibjpeg-libjpeg (from versions: none)
ERROR: No matching distribution found for pylibjpeg-libjpeg


**Conclusion.** Two T4 GPUs are available.

The same overview of the training data as in notebook 01. The training tables are still needed: the side of the knee is learned from the training studies in the laterality step below.

In [2]:
tr = pd.read_csv(f"{R}/train.csv", encoding_errors="replace")
trs = pd.read_csv(f"{R}/train_series.csv")
gold = tr[tr[LAB[0]].notna()]
print(f"{len(tr)} studies, {len(gold)} with gold labels, {len(trs)} series")
print(gold[LAB].mean().round(2).to_string())
print(trs.Anatomical_Plane.value_counts().to_dict(), trs.groupby("StudyInstanceUID").size().describe()[["mean", "max"]].to_dict())
print(tr.Report.str.len().describe()[["mean", "50%", "max"]].round().to_dict())

4407 studies, 58 with gold labels, 24371 series
ACL                 0.41
MCL                 0.16
Medial Meniscus     0.45
Lateral Meniscus    0.40
Medial OA           0.26
Lateral OA          0.19
PF OA               0.36
Effusion            0.60
Synovitis           0.47
Baker's             0.21
Contusion           0.33
Fracture            0.31
{'Sagittal': 9864, 'Coronal': 8609, 'Axial': 5898} {'mean': 5.530065804402088, 'max': 14.0}
{'mean': 1098.0, '50%': 977.0, 'max': 4743.0}


**Conclusion.** Same data as in notebooks 01 and 02.

The volume builder from notebook 02. The training array is read from notebook 02's output; the functions are needed to build the test volumes in the last cell.

In [3]:
SLOTS = [("Sagittal", 1, 20), ("Sagittal", 0, 12), ("Coronal", 1, 16), ("Coronal", 0, 10), ("Axial", 1, 14)]
NS = sum(k for _, _, k in SLOTS)
PREPS = {"v2": (192, 150), "v3": (224, 140), "v4": (320, 130)}
BUILD_SLOTS, PART = None, ""
AXIS = {"Sagittal": 0, "Coronal": 1, "Axial": 2}
TARGET = {"Sagittal": ((0, 1, 0), (0, 0, -1)), "Coronal": ((1, 0, 0), (0, 0, -1)), "Axial": ((1, 0, 0), (0, 1, 0))}


def canon_op(r, c, plane):
    tr, tc = (np.array(v, float) for v in TARGET[plane])
    best = None
    for t in (0, 1):
        for fu in (0, 1):
            for fl in (0, 1):
                r1, c1 = (c, r) if t else (r, c)
                score = (-c1 if fu else c1) @ tc + (-r1 if fl else r1) @ tr
                if best is None or score > best[0]:
                    best = (score, t, fu, fl)
    return best[1:]


def canonicalise(a, op):
    t, fu, fl = op
    a = a.T if t else a
    a = a[::-1] if fu else a
    return a[:, ::-1] if fl else a


def crop_square(a, ps_r, ps_c, s, fov):
    hp, wp = max(int(round(fov / ps_r)), 8), max(int(round(fov / ps_c)), 8)
    h, w = a.shape
    y0, x0 = (h - hp) // 2, (w - wp) // 2
    sy0, sx0, sy1, sx1 = max(y0, 0), max(x0, 0), min(y0 + hp, h), min(x0 + wp, w)
    out = np.zeros((hp, wp), np.float32)
    out[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = a[sy0:sy1, sx0:sx1]
    return cv2.resize(out, (s, s), interpolation=cv2.INTER_AREA if max(hp, wp) > s else cv2.INTER_LINEAR)


def load_slot(path, plane, k, s, fov):
    files = glob.glob(os.path.join(path, "*.dcm"))
    hdrs = [pydicom.dcmread(f, stop_before_pixels=True) for f in files]
    h0 = hdrs[0]
    iop = np.array(h0.ImageOrientationPatient, float)
    ps = [float(x) for x in getattr(h0, "PixelSpacing", [fov / max(int(h0.Rows), 1)] * 2)]
    try:
        key = [float(h.ImagePositionPatient[AXIS[plane]]) for h in hdrs]
    except Exception:
        key = [float(getattr(h, "InstanceNumber", i)) for i, h in enumerate(hdrs)]
    order = np.argsort(key)
    idx = order[np.linspace(0.04 * (len(order) - 1), 0.96 * (len(order) - 1), k).round().astype(int)]
    op = canon_op(iop[:3], iop[3:], plane)
    ps_r, ps_c = (ps[1], ps[0]) if op[0] else (ps[0], ps[1])
    vol = np.stack([crop_square(canonicalise(pydicom.dcmread(files[i]).pixel_array.astype(np.float32), op), ps_r, ps_c, s, fov) for i in idx])
    lo, hi = np.percentile(vol, [1, 99.5])
    return (np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)


def assign_slots(d):
    used, out = set(), []
    for plane, fluid, _ in SLOTS:
        c = d[(d.Anatomical_Plane == plane) & ~d.SeriesInstanceUID.isin(used)]
        pick = c[c.Fluid_Sensitive == fluid]
        if len(pick) == 0 and fluid == 1:
            pick = c
        if len(pick) == 0:
            out.append(None)
            continue
        sid = pick.sort_values("Fat_Suppression", ascending=False, kind="stable").SeriesInstanceUID.iloc[0]
        used.add(sid)
        out.append(sid)
    return out


def _study(root, prep, sel, a):
    study, sids = a
    s, fov = PREPS[prep]
    vol = np.zeros((sum(SLOTS[j][2] for j in sel), s, s), np.uint8)
    o = 0
    for j, ((plane, _, k), sid) in enumerate(zip(SLOTS, sids)):
        if j not in sel:
            sids = [None if n == j else x for n, x in enumerate(sids)]
            continue
        if sid is not None:
            try:
                vol[o:o + k] = load_slot(f"{root}/{study}/{sid}", plane, k, s, fov)
            except Exception as e:
                print("fail", study, sid, str(e)[:80], flush=True)
                sids = [None if x == sid else x for x in sids]
        o += k
    return vol, sids


def build_volumes(ts, root, out, studies=None, prep=PREP, sel=None):
    sel = list(range(len(SLOTS))) if sel is None else sel
    groups = dict(tuple(ts.groupby("StudyInstanceUID")))
    studies = studies or sorted(groups)
    jobs = [(s, assign_slots(groups[s]) if s in groups else [None] * len(SLOTS)) for s in studies]
    arr = np.lib.format.open_memmap(out, "w+", np.uint8, (len(jobs), sum(SLOTS[j][2] for j in sel)) + (PREPS[prep][0],) * 2)
    rows = []
    with Pool(4) as p:
        for i, (v, sids) in enumerate(p.imap(partial(_study, root, prep, sel), jobs, chunksize=4)):
            arr[i] = v
            rows.append({"StudyInstanceUID": jobs[i][0], **{f"sid{j}": x for j, x in enumerate(sids)}})
    arr.flush()
    tab = pd.DataFrame(rows)
    tab.to_csv(out.replace(".npy", ".csv"), index=False)
    return tab


class Split:
    def __init__(self, parts):
        self.parts = parts
        self.cut = np.cumsum([0] + [p.shape[1] for p in parts])
        self.shape = (parts[0].shape[0], self.cut[-1]) + parts[0].shape[2:]

    def __len__(self):
        return self.shape[0]

    def __getitem__(self, k):
        if isinstance(k, tuple):
            p = np.searchsorted(self.cut, k[1], side="right") - 1
            return self.parts[p][k[0], k[1] - self.cut[p]]
        return np.concatenate([p[k] for p in self.parts], axis=1 if np.ndim(k) else 0)


cached = find(f"train_{PREP}.npy")
parts = sorted(find(f"train_{PREP}?.npy"))
if cached:
    TRV = np.load(cached[0], mmap_mode="r")
    trtab = pd.read_csv(cached[0].replace(".npy", ".csv"))
elif parts and not BUILD_SLOTS:
    TRV = Split([np.load(p, mmap_mode="r") for p in parts])
    tabs = [pd.read_csv(p.replace(".npy", ".csv")) for p in parts]
    trtab = tabs[0]
    for t in tabs[1:]:
        trtab = trtab.combine_first(t)
else:
    trtab = build_volumes(trs, f"{R}/train_series", f"/kaggle/working/train_{PREP}{PART}.npy", sel=BUILD_SLOTS)
    TRV = np.load(f"/kaggle/working/train_{PREP}{PART}.npy", mmap_mode="r")
STUDIES = list(trtab.StudyInstanceUID)
print(TRV.shape, trtab[[f"sid{j}" for j in range(len(SLOTS))]].notna().mean().round(2).to_dict())
SLOT_OFF = np.cumsum([0] + [k for _, _, k in SLOTS])


def load_rows(arr, lat, rows):
    out = np.ascontiguousarray(arr[np.asarray(rows)])
    for n, i in enumerate(rows):
        if lat[i] == 1:
            for j in (0, 1):
                out[n, SLOT_OFF[j]:SLOT_OFF[j + 1]] = out[n, SLOT_OFF[j]:SLOT_OFF[j + 1]][::-1]
            out[n, SLOT_OFF[2]:] = out[n, SLOT_OFF[2]:, :, ::-1]
    return out

(4407, 72, 224, 224) {'sid0': 1.0, 'sid1': 0.97, 'sid2': 1.0, 'sid3': 0.74, 'sid4': 1.0}


**Conclusion.** The same array as in notebook 02: 4,407 x 72 x 224 x 224, with the same slot coverage.

Which knee is it? The DICOM laterality tag is filled for half of the studies. A ResNet34 with ImageNet weights trains for three epochs on six axial slices of the tagged training studies, holding out the studies from three scanner makes and a random 12% of the rest. For an untagged study its logit is combined with the knee's position in patient coordinates, which is reliable for five scanner makes. The tag wins whenever it is present. The test studies get the same treatment in the last cell.

In [4]:
GEO_OK = ("SIEM", "PHIL", "GE M", "TOSH", "CANO")
AX0, LS = sum(k for _, _, k in SLOTS[:4]), 128


def _hdr(root, a):
    try:
        h = pydicom.dcmread(sorted(glob.glob(f"{root}/{a[0]}/{a[1]}/*.dcm"))[0], stop_before_pixels=True)
        iop, ps = np.array(h.ImageOrientationPatient, float), [float(x) for x in h.PixelSpacing]
        ctr = np.array(h.ImagePositionPatient, float) + iop[:3] * ps[1] * (h.Columns - 1) / 2 + iop[3:] * ps[0] * (h.Rows - 1) / 2
        return str(getattr(h, "Laterality", "")).upper(), str(getattr(h, "Manufacturer", "")).upper()[:4], ctr[0]
    except Exception:
        return "", "", np.nan


def study_agg(d, studies):
    d = d.assign(tag=d.lat.map({"R": 0, "RIGHT": 0, "L": 1, "LEFT": 1}))
    first = lambda x: next((v for v in x if v), "")
    st = d.groupby("StudyInstanceUID").agg(tag=("tag", lambda x: x.dropna().mode().iloc[0] if x.notna().any() else np.nan),
                                           mk=("mk", first), cx=("cx", "median"))
    return st.reindex(studies)


def study_headers(ts, root, studies):
    with Pool(4) as p:
        res = p.map(partial(_hdr, root), list(zip(ts.StudyInstanceUID, ts.SeriesInstanceUID)), chunksize=32)
    return study_agg(pd.DataFrame(res, columns=["lat", "mk", "cx"]).assign(StudyInstanceUID=ts.StudyInstanceUID.values), studies)


def lat_images(arr, rows):
    return np.stack([[cv2.resize(arr[i, AX0 + 4 + j], (LS, LS), interpolation=cv2.INTER_AREA) for j in range(6)] for i in rows])


class LatNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.bb = timm.create_model("resnet34", pretrained=False, num_classes=0)
        if pretrained:
            self.bb.load_state_dict(torch.load(find("resnet34.pt")[0], map_location="cpu", weights_only=True))
        self.fc = nn.Linear(self.bb.num_features, 1)

    def forward(self, x):
        return self.fc(self.bb(x.expand(-1, 3, -1, -1))).squeeze(1)


def lat_logits(net, arr, n):
    net.eval()
    out = np.zeros(n)
    with torch.no_grad():
        for i in range(0, n, 32):
            xb = torch.from_numpy(lat_images(arr, range(i, min(i + 32, n)))).reshape(-1, 1, LS, LS).to(DEV)
            out[i:i + 32] = net(((xb.float() / 255) - 0.3) / 0.3).reshape(-1, 6).mean(1).float().cpu().numpy()
    return out


def lat_decide(st, logit, has_ax):
    geo = np.where(st.mk.isin(GEO_OK), np.sign(st.cx.fillna(0).values), 0)
    z = np.clip(logit, -8, 8) * has_ax + 3 * geo
    return np.where(st.tag.notna(), st.tag, (z > 0).astype(float))


def train_lat(arr, st, has_ax, epochs=3):
    rs = np.random.RandomState(0)
    tagged = st.tag.notna().values & has_ax
    hold = tagged & (st.mk.isin(["GEHC", "FUJI", "HITA"]).values | (rs.rand(len(st)) < 0.12))
    tr_rows = np.where(tagged & ~hold)[0]
    net, y = LatNet().to(DEV), st.tag.fillna(0).values.astype(np.float32)
    opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-2)
    LX = lat_images(arr, tr_rows)
    for ep in range(epochs):
        net.train()
        perm = rs.permutation(len(tr_rows))
        for i in range(0, len(perm), 16):
            b = perm[i:i + 16]
            xb = torch.from_numpy(LX[b, rs.randint(0, 6, len(b))]).unsqueeze(1).float() / 255
            xb = (torch.roll(xb, rs.randint(-8, 9, 2).tolist(), (2, 3)) * (1 + (torch.rand(len(b), 1, 1, 1) - .5) * 0.4) - 0.3) / 0.3
            loss = F.binary_cross_entropy_with_logits(net(xb.to(DEV)), torch.from_numpy(y[tr_rows[b]]).to(DEV))
            opt.zero_grad()
            loss.backward()
            opt.step()
    acc = ((lat_logits(net, arr, len(st))[hold] > 0) == (st.tag.values[hold] == 1)).mean()
    print(f"laterality holdout acc {acc:.4f} on {hold.sum()}", flush=True)
    return net


geo = find("geom.csv")
if geo:
    gm = pd.read_csv(geo[0], usecols=["StudyInstanceUID", "lat", "man", "cx"])
    gm = gm.assign(lat=gm.lat.fillna("").astype(str).str.upper(), mk=gm.man.fillna("").astype(str).str.upper().str[:4])
    sth = study_agg(gm, STUDIES)
else:
    sth = study_headers(trs, f"{R}/train_series", STUDIES)
has_ax = trtab.sid4.notna().values
LNET = train_lat(TRV, sth, has_ax)
LATF = lat_decide(sth, lat_logits(LNET, TRV, len(STUDIES)), has_ax)
print("left share", LATF.mean().round(3), "tag coverage", sth.tag.notna().mean().round(3))

laterality holdout acc 0.9915 on 353
left share 0.469 tag coverage 0.5


**Conclusion.** The image model alone is right on about 99% of 353 held-out tagged studies; the exact figure varies a little between runs because the model is retrained each time. About 47% of the training studies are called left, and half carry the tag.

The training labels, as in notebook 04. They are not used for prediction, but the cell also prints how each label source scores against the radiologist labels.

In [5]:
SYSTEM = """You read a knee MRI radiology report (any language) and label 12 findings, 1 = present, 0 = absent or not mentioned.
ACL: anterior cruciate ligament tear, rupture or sprain (partial or complete); not mere mild degeneration.
MCL: medial collateral ligament tear or sprain of any grade.
Medial Meniscus / Lateral Meniscus: tear of that meniscus (includes radial, root, bucket handle, complex, horizontal, displaced flap); ignore intact meniscus, degeneration only or grade 1-2 signal without tear.
Medial OA: osteoarthritis or cartilage loss / chondropathy / chondrosis / osteophytes in the medial femorotibial compartment.
Lateral OA: same for the lateral femorotibial compartment.
PF OA: chondropathy, cartilage defect or osteoarthritis of the patellofemoral joint (patella or trochlea).
Effusion: joint effusion, hydrops, excess joint fluid.
Synovitis: synovitis, synovial thickening or proliferation.
Baker's: popliteal or Baker's cyst.
Contusion: bone contusion, bone bruise, marrow or bone edema.
Fracture: fracture, insufficiency fracture, osteochondral or impaction fracture.
Answer with exactly 12 lines in this order, each 'Name: 0' or 'Name: 1', nothing else."""


def label_reports(tr, path, tok_budget=13000):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(path, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float16, device_map="auto").eval()
    reps = [str(r)[:3500] for r in tr.Report]
    ids = [len(tok.encode(r)) for r in reps]
    d0, d1 = tok.encode("0")[0], tok.encode("1")[0]
    batches, cur = [], []
    for i in np.argsort(ids):
        if cur and ((len(cur) + 1) * (min(ids[i], 1800) + 500) > tok_budget or len(cur) >= 24):
            batches.append(cur)
            cur = []
        cur.append(i)
    batches.append(cur)
    out = np.full((len(tr), 12), np.nan, np.float32)
    t0 = time.time()
    for n, idx in enumerate(batches):
        chats = [tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": reps[i]}],
                                         tokenize=False, add_generation_prompt=True, enable_thinking=False) for i in idx]
        enc = tok(chats, return_tensors="pt", padding=True).to("cuda:0")
        with torch.no_grad():
            g = model.generate(**enc, max_new_tokens=90, do_sample=False, output_scores=True, return_dict_in_generate=True)
        seq, sc = g.sequences[:, enc.input_ids.shape[1]:], torch.stack(g.scores, 1).float()
        for r, i in enumerate(idx):
            p = [torch.softmax(sc[r, t, [d0, d1]], 0)[1].item() for t, tk in enumerate(seq[r].tolist()) if tk in (d0, d1)]
            if len(p) == 12:
                out[i] = p
        if n % 20 == 0:
            print(n, len(batches), round(time.time() - t0), flush=True)
    del model
    torch.cuda.empty_cache()
    return pd.DataFrame(out, columns=LAB).assign(StudyInstanceUID=tr.StudyInstanceUID.values)


cached = find("pseudo.csv")
if cached:
    PS = pd.read_csv(cached[0])
else:
    qwen = os.path.dirname(find("qwen-3/transformers/8b/*/config.json")[0])
    PS = label_reports(tr, qwen)
    PS.to_csv("/kaggle/working/pseudo.csv", index=False)
PUBLIC_LABELS = ["llm_labels_v4_blend.csv", "report_labels_v2.csv", "report_labels_gpt56sol.csv"]


def label_table(d):
    d = d.drop_duplicates("StudyInstanceUID").set_index("StudyInstanceUID")[LAB].apply(pd.to_numeric, errors="coerce")
    d = d.reindex(tr.StudyInstanceUID)
    return d.fillna(d.mean())


SRC = {"qwen3-8b (ours)": label_table(PS)}
for name in PUBLIC_LABELS:
    fs = find(name)
    if fs:
        SRC[name] = label_table(pd.concat([pd.read_csv(f) for f in fs]))
FUSED = sum(SRC.values()) / len(SRC)
g = tr[tr[LAB[0]].notna()].set_index("StudyInstanceUID")
gold_auc = lambda d: np.mean([roc_auc_score(g[c], d[:, k]) for k, c in enumerate(LAB)])
for name, d in list(SRC.items()) + [("fused", FUSED)]:
    print(f"{name:28s} gold macro AUC {gold_auc(d.loc[g.index, LAB].values):.4f}")

oofs = [f for t in TEACHERS for f in find(f"oof_{t}_*.csv")]
if oofs:
    T = pd.concat([pd.read_csv(f) for f in oofs]).dropna().groupby("StudyInstanceUID")[LAB].mean().reindex(tr.StudyInstanceUID)
    GT = np.mean([p for t in TEACHERS for f in find(f"gold_{t}_*.npy") for p in np.load(f)], 0)
    GT = pd.DataFrame(GT, index=[s for s in STUDIES if s in g.index], columns=LAB).loc[g.index].values
    for b in (0, .25, .5, .75, 1):
        print(f"distill beta {b}: gold AUC {gold_auc((1 - b) * FUSED.loc[g.index, LAB].values + b * GT):.4f}")
    print(f"teacher OOF coverage {T[LAB[0]].notna().mean():.3f}, using beta {DISTILL}")
    FUSED = FUSED.where(T.isna(), (1 - DISTILL) * FUSED + DISTILL * T)

qwen3-8b (ours)              gold macro AUC 0.8570
llm_labels_v4_blend.csv      gold macro AUC 0.8927
report_labels_v2.csv         gold macro AUC 0.8633
report_labels_gpt56sol.csv   gold macro AUC 0.8352
fused                        gold macro AUC 0.8972


**Conclusion.** The fused labels score 0.897 against the radiologist labels, as in notebook 04.

The model definition and the fold loop. With `TRAIN = False` every fold model is loaded from the inputs instead of trained, and no training happens.

In [6]:
NSLOT, KS = len(SLOTS), [k for _, _, k in SLOTS]
SLOT_OF = np.repeat(np.arange(NSLOT), KS)
POS_OF = np.concatenate([np.arange(k) / max(k - 1, 1) for k in KS]).astype(np.float32)
NORM = {"": (0.45, 0.225), "rad": (0.5, 0.5)}


def parse_arch(arch):
    name, _, tag = arch.partition("@")
    tag, _, opt = tag.partition("+")
    return name, tag, opt


def lat_for(arch, lat):
    return np.zeros_like(lat) if "n" in parse_arch(arch)[2] else lat


def weight_file(arch):
    name, tag, _ = parse_arch(arch)
    return f"{name}_{tag}.pt" if tag else f"{name}.pt"


class Net(nn.Module):
    def __init__(self, arch=ARCH, pretrained=True, dim=256, res=RES):
        super().__init__()
        name, tag, opt = parse_arch(arch)
        self.mean, self.std = NORM[tag]
        self.bb = timm.create_model(name, pretrained=False, num_classes=0, **({"img_size": res} if name.startswith("vit") else {}))
        if pretrained:
            w = find(weight_file(arch))
            assert w, f"missing pretrained weights {weight_file(arch)}"
            self.bb.load_state_dict(torch.load(w[0], map_location="cpu", weights_only=True))
            if "c" in opt:
                conv = next(m for m in self.bb.modules() if isinstance(m, nn.Conv2d) and m.in_channels == 3)
                with torch.no_grad():
                    centre = conv.weight.sum(1)
                    conv.weight.zero_()
                    conv.weight[:, 1] = centre
        self.spatial = "s" in opt
        nf = self.bb.num_features
        if self.spatial:
            self.sp = nn.Conv2d(nf, 1, 1)
        self.proj = nn.Sequential(nn.Linear(nf * (2 if self.spatial else 1), dim), nn.LayerNorm(dim))
        self.slot = nn.Embedding(NSLOT, dim)
        self.pos = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))
        layer = nn.TransformerEncoderLayer(dim, 4, 2 * dim, 0.1, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.att = nn.Linear(dim, 12)
        self.out_w = nn.Parameter(torch.randn(12, dim) * 0.02)
        self.out_b = nn.Parameter(torch.zeros(12))
        self.drop = nn.Dropout(0.2)

    def forward(self, x, slot, pos, tm):
        b, t = x.shape[:2]
        x = (x.flatten(0, 1) - self.mean) / self.std
        if self.spatial:
            fm = self.bb.forward_features(x)
            a = torch.softmax(self.sp(fm).flatten(2).float(), -1).to(fm.dtype)
            feat = torch.cat([fm.mean((2, 3)), (fm.flatten(2) * a).sum(-1)], 1)
        else:
            feat = self.bb(x)
        f = self.proj(feat).view(b, t, -1)
        f = f + self.slot(slot) + self.pos(pos.unsqueeze(-1).to(f.dtype))
        f = self.enc(f, src_key_padding_mask=~tm)
        a = torch.softmax(self.att(f).float().masked_fill(~tm.unsqueeze(-1), -1e4), 1)
        z = torch.einsum("btk,btd->bkd", a, f.float())
        return (self.drop(z) * self.out_w).sum(-1) + self.out_b


class Vols(torch.utils.data.Dataset):
    def __init__(self, arr, lat, rows):
        self.arr, self.lat, self.rows = arr, lat, rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return torch.from_numpy(load_rows(self.arr, self.lat, [self.rows[i]])[0])


def loader(arr, lat, rows, bs, shuffle):
    return torch.utils.data.DataLoader(Vols(arr, lat, rows), bs, shuffle=shuffle, num_workers=0 if HIDDEN else 4, drop_last=shuffle,
                                       pin_memory=False, persistent_workers=False)


def windows(n, rng=None):
    if rng is None:
        c = np.arange(SLOT_OFF[-1])
        slot, pos = SLOT_OF, POS_OF
        lo, hi = SLOT_OFF[slot], SLOT_OFF[slot + 1] - 1
        idx = np.clip(c[:, None] + np.arange(-1, 2), lo[:, None], hi[:, None])
        return (np.broadcast_to(idx, (n,) + idx.shape), np.broadcast_to(slot, (n, len(slot))), np.broadcast_to(pos, (n, len(slot))))
    idx, slot, pos = [], [], []
    for j, (k, m) in enumerate(zip(KS, WIN)):
        if STRAT:
            c = np.minimum(((np.arange(m) + rng.rand(n, m)) * k / m).astype(int), k - 1)
        else:
            c = np.stack([rng.choice(k, m, replace=k < m) for _ in range(n)])
        idx.append(SLOT_OFF[j] + np.clip(c[..., None] + np.arange(-1, 2), 0, k - 1))
        slot.append(np.full((n, m), j))
        pos.append(c / max(k - 1, 1))
    return np.concatenate(idx, 1), np.concatenate(slot, 1), np.concatenate(pos, 1).astype(np.float32)


def batch_tokens(v, mask, rng=None, aug=False, res=RES):
    n = v.shape[0]
    idx, slot, pos = (torch.from_numpy(np.ascontiguousarray(a)).to(v.device) for a in windows(n, rng))
    x = v[torch.arange(n, device=v.device)[:, None, None], idx]
    t = x.shape[1]
    x = x.flatten(0, 1).float().div(255)
    if aug:
        u = lambda: torch.rand(len(x), device=x.device) - .5
        ang, sc = u() * 2 * math.radians(12), 1 + u() * 0.25
        th = torch.stack([torch.stack([sc * torch.cos(ang), -sc * torch.sin(ang), u() * 0.2], 1),
                          torch.stack([sc * torch.sin(ang), sc * torch.cos(ang), u() * 0.2], 1)], 1)
        x = F.grid_sample(x, F.affine_grid(th, (len(x), 3, res, res), align_corners=False), align_corners=False)
        x = x * (1 + u().view(-1, 1, 1, 1) * 0.4) + u().view(-1, 1, 1, 1) * 0.2
    else:
        x = F.interpolate(x, size=(res, res), mode="bilinear", align_corners=False)
    tm = torch.from_numpy(mask).to(v.device)[torch.arange(n, device=v.device)[:, None], slot]
    tm[:, 0] |= ~tm.any(1)
    return x.view(n, t, 3, res, res), slot, pos, tm


def predict(model, arr, lat, mask, rows, bs=4, draws=0, res=RES):
    model.eval()
    rng = np.random.RandomState(0)
    out = []
    with torch.no_grad(), torch.autocast(DEV, torch.float16, enabled=DEV == "cuda"):
        for i, v in enumerate(loader(arr, lat, rows, bs, False)):
            r, v = rows[i * bs:(i + 1) * bs], v.to(DEV)
            if draws:
                p = np.mean([torch.sigmoid(model(*batch_tokens(v, mask[r], rng, res=res)).float()).cpu().numpy() for _ in range(draws)], 0)
            else:
                xb, slot, pos, tm = batch_tokens(v, mask[r], res=res)
                p = torch.sigmoid(model(xb, slot, pos, tm).float()).cpu().numpy()
                if TTA_REV:
                    p = (p + torch.sigmoid(model(xb.flip(2), slot, pos, tm).float()).cpu().numpy()) / 2
            out.append(p)
    return np.concatenate(out)


def mean_auc(y, p):
    return np.mean([roc_auc_score(y[:, k], p[:, k]) for k in range(y.shape[1]) if len(set(y[:, k])) > 1])


def train_model(seed, Y, weight, train_rows, val_rows=None, gold_rows=None, arch=ARCH):
    torch.manual_seed(seed)
    rng = np.random.RandomState(seed)
    net = Net(arch).to(DEV)
    lat = lat_for(arch, LATF)
    lr_bb = LR_ARCH.get(arch, LR_BB)
    ema = torch.optim.swa_utils.AveragedModel(net, multi_avg_fn=torch.optim.swa_utils.get_ema_multi_avg_fn(0.998), use_buffers=True)
    model = nn.DataParallel(net) if torch.cuda.device_count() > 1 else net
    bbp = list(net.bb.parameters())
    ids = {id(p) for p in bbp}
    opt = torch.optim.AdamW([{"params": bbp, "lr": lr_bb}, {"params": [p for p in net.parameters() if id(p) not in ids], "lr": LR_HEAD}], weight_decay=1e-2)
    steps = EPOCHS * (len(train_rows) // BS)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, [lr_bb, LR_HEAD], total_steps=steps, pct_start=0.07)
    scaler = torch.amp.GradScaler(enabled=DEV == "cuda")
    t0 = time.time()
    for ep in range(EPOCHS):
        model.train()
        perm = rng.permutation(train_rows)
        tl, nb = 0, 0
        for i, v in enumerate(loader(TRV, lat, perm, BS, False)):
            if (i + 1) * BS > len(perm):
                break
            b = perm[i * BS:(i + 1) * BS]
            xb, slot, pos, tm = batch_tokens(v.to(DEV, non_blocking=True), MASK[b], rng, aug=True)
            yb, wb = torch.from_numpy(Y[b]).to(DEV), torch.from_numpy(weight[b]).to(DEV)
            with torch.autocast(DEV, torch.float16, enabled=DEV == "cuda"):
                lo = model(xb, slot, pos, tm)
            loss = (F.binary_cross_entropy_with_logits(lo.float(), yb, reduction="none").mean(1) * wb).mean()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(net.parameters(), 2.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            ema.update_parameters(net)
            tl, nb = tl + loss.item(), nb + 1
        msg = f"seed {seed} ep {ep} loss {tl / max(nb, 1):.4f} t {time.time() - t0:.0f} {ram()}"
        if ep % 4 == 3 or ep == EPOCHS - 1:
            if len(val_rows):
                msg += f" val {mean_auc((Y[val_rows] > .5).astype(int), predict(ema.module, TRV, lat, MASK, val_rows)):.4f}"
            msg += f" gold {mean_auc(GOLD_Y, predict(ema.module, TRV, lat, MASK, gold_rows)):.4f}"
        print(msg, flush=True)
    return ema.module


MASK = trtab[[f"sid{j}" for j in range(len(SLOTS))]].notna().values
trg = tr.set_index("StudyInstanceUID").loc[STUDIES]
is_gold = trg[LAB[0]].notna().values
Y = FUSED.loc[STUDIES, LAB].values.astype(np.float32)
GOLD_Y = trg.loc[is_gold, LAB].values.astype(int)
weight = np.ones(len(STUDIES), np.float32)
gold_rows = np.where(is_gold)[0]
pool = np.where(~is_gold)[0]
if LIMIT:
    pool = pool[:LIMIT]
folds = list(KFold(FOLDS, shuffle=True, random_state=SEED).split(pool))
OOF = np.full((len(STUDIES), 12), np.nan, np.float32)
GP = []
for f in RUN_FOLDS:
    tr_rows, va_rows = (pool, pool[:0]) if f == FOLDS else (pool[folds[f][0]], pool[folds[f][1]])
    w = find(f"own_{FAM}_f{f}.pt")
    if w:
        sd = torch.load(w[0], map_location="cpu", weights_only=True)
    else:
        assert TRAIN, f"missing weights own_{FAM}_f{f}.pt"
        m = train_model(SEED + f, Y, weight, tr_rows, va_rows, gold_rows, ARCH)
        sd = {k: v.cpu() for k, v in m.state_dict().items()}
        torch.save(sd, f"own_{FAM}_f{f}.pt")
        del m
        torch.cuda.empty_cache()
    if w and not EVAL_CACHED:
        continue
    net = Net(pretrained=False).to(DEV)
    net.load_state_dict(sd)
    GP.append(predict(net, TRV, lat_for(ARCH, LATF), MASK, gold_rows, draws=EVAL_DRAWS))
    if f == FOLDS:
        print(f"full fit gold {mean_auc(GOLD_Y, GP[-1]):.4f}", flush=True)
        np.save(f"gold_{FAM}_full.npy", np.array(GP[-1:]))
        del net
        continue
    OOF[va_rows] = predict(net, TRV, lat_for(ARCH, LATF), MASK, va_rows, draws=EVAL_DRAWS)
    if f == RUN_FOLDS[0] and not w:
        alt = predict(net, TRV, lat_for(ARCH, LATF), MASK, va_rows, draws=0 if EVAL_DRAWS else 4)
        print(f"fold {f} val with draws={0 if EVAL_DRAWS else 4}: {mean_auc((Y[va_rows] > .5).astype(int), alt):.4f}", flush=True)
    del net
    print(f"fold {f} val {mean_auc((Y[va_rows] > .5).astype(int), OOF[va_rows]):.4f} gold {mean_auc(GOLD_Y, GP[-1]):.4f}", flush=True)
done = ~np.isnan(OOF[:, 0])
if done.any():
    print(f"OOF auc vs fused labels {mean_auc((Y[done] > .5).astype(int), OOF[done]):.4f}  gold (fold mean) {mean_auc(GOLD_Y, np.mean(GP, 0)):.4f}")
    print("per-label gold", dict(zip(LAB, np.round([roc_auc_score(GOLD_Y[:, k], np.mean(GP, 0)[:, k]) for k in range(12)], 3))))
    np.save(f"gold_{FAM}_{'-'.join(map(str, RUN_FOLDS))}.npy", np.array(GP))
    pd.DataFrame(OOF, columns=LAB).assign(StudyInstanceUID=STUDIES).to_csv(f"oof_{FAM}_{'-'.join(map(str, RUN_FOLDS))}.csv", index=False)

for fam in SUB_TRAIN:
    assert FAMILIES[fam][1:] == (PREP, RES), f"{fam} needs prep {FAMILIES[fam][1]}"
    assert find(weight_file(FAMILIES[fam][0])), f"missing pretrained weights for {fam}"
for fam in SUB_TRAIN if HIDDEN else []:
    if find(f"own_{fam}_f{FOLDS}.pt"):
        continue
    if (time.time() - T0) / 3600 + FIT_H.get(fam, 2.0) > 7.5 or ram_gb()[0] > RAM_GUARD:
        print("skip", fam, "budget", ram(), flush=True)
        break
    try:
        Yf, wf = Y.copy(), weight.copy()
        Yf[gold_rows], wf[gold_rows] = GOLD_Y, 2.0
        m = train_model(SEED + FOLDS, Yf, wf, np.concatenate([pool, gold_rows]), pool[:0], gold_rows, FAMILIES[fam][0])
        torch.save({k: v.cpu() for k, v in m.state_dict().items()}, f"own_{fam}_f{FOLDS}.pt")
        del m
    except Exception as e:
        print("full fit failed", fam, repr(e)[:300], flush=True)
    gc.collect()
    torch.cuda.empty_cache()
    print("after", fam, ram(), flush=True)

Prediction. The test volumes are built with the code from notebook 02, each test knee's side is decided, and every fold model predicts with one 3-slice window centred on every slice. The five folds of a family are averaged as probabilities; the three families are combined by averaging per-finding ranks, which is what ROC AUC rewards. Studies with no usable series get 0.5.

In [7]:
sub = pd.read_csv(f"{R}/sample_submission.csv")
te = pd.read_csv(f"{R}/test_series.csv")
tstudies = list(sub.StudyInstanceUID)
teh = study_headers(te, f"{R}/test_series", tstudies)
TEST = {}
FAM_W = {}


def fhash(p):
    with open(p, "rb") as fh:
        fh.seek(os.path.getsize(p) // 2)
        return hashlib.md5(fh.read(1 << 20)).hexdigest()


for fam, (arch, prep, res) in FAMILIES.items():
    ws = list({fhash(w): w for f in range(FOLDS + 1) for w in find(f"own_{fam}_f{f}.pt")}.values())
    if not ws:
        print("no weights for", fam)
        continue
    if prep not in TEST:
        tab = build_volumes(te, f"{R}/test_series", f"/kaggle/working/test_{prep}.npy", studies=tstudies, prep=prep)
        TEST[prep] = (np.load(f"/kaggle/working/test_{prep}.npy", mmap_mode="r"), tab[[f"sid{j}" for j in range(len(SLOTS))]].notna().values)
    tev, tmask = TEST[prep]
    if "TLAT" not in globals():
        TLAT = lat_decide(teh, lat_logits(LNET, tev, len(tstudies)), tmask[:, 4])
    p = np.zeros((len(tstudies), 12))
    for w in ws:
        net = Net(arch, pretrained=False, res=res).to(DEV)
        net.load_state_dict(torch.load(w, map_location="cpu", weights_only=True))
        p += predict(net, tev, lat_for(arch, TLAT), tmask, np.arange(len(tstudies)), draws=EVAL_DRAWS, res=res) / len(ws)
        del net
    FAM_W[fam] = (p, len(ws))
    print(fam, len(ws), "folds", p.mean(0).round(3), ram(), flush=True)
assert FAM_W, "no trained models found"
if len(FAM_W) == 1:
    pred = next(iter(FAM_W.values()))[0]
else:
    wt = {f: FAM_WEIGHT.get(f, 1.0) for f in FAM_W}
    pred = sum(wt[f] * pd.DataFrame(p).rank(pct=True).values for f, (p, _) in FAM_W.items()) / sum(wt.values())
    print("family weights", wt)
tmask_any = np.any([m.any(1) for _, m in TEST.values()], 0)
pred[~tmask_any] = 0.5
sub[LAB] = pred
sub.to_csv("submission.csv", index=False)
print(sub[LAB].describe().T[["mean", "min", "max"]])

no weights for resnet50
no weights for r50v3
no weights for coat0
no weights for r50v4
no weights for eb3v3
no weights for sx50v3
no weights for dinosv3
no weights for sx50r2
no weights for r50r2
no weights for radr50v3
no weights for radr2
no weights for rdn121
no weights for radc
no weights for rdn121c
no weights for sx50n
no weights for radcn
radcs 5 folds [0.112 0.116 0.511 0.121 0.644 0.405 0.749 0.455 0.221 0.429 0.232 0.13 ] ram 3.0/31.3G
no weights for radc4
no weights for sx50s
no weights for radcsn
no weights for radc2
no weights for radcs2
no weights for radcn2
radcr 5 folds [0.106 0.107 0.511 0.113 0.614 0.358 0.715 0.412 0.226 0.433 0.239 0.127] ram 3.0/31.3G
sx50r 5 folds [0.164 0.085 0.408 0.082 0.616 0.352 0.687 0.328 0.216 0.37  0.279 0.135] ram 3.0/31.3G
family weights {'radcs': 1.0, 'radcr': 1.0, 'sx50r': 1.0}
                      mean       min       max
ACL               0.666667  0.444444  1.000000
MCL               0.666667  0.333333  0.888889
Medial Meniscus   

**Conclusion.** All three families are found with five folds each, and the submission has one row per test study. With three example studies the ranks can only be 1/3, 2/3 and 1, so the summary only shows that the file is complete; the real check is the leaderboard score of the scoring run.